# E04 – PyTorch alapok
**Generatív AI és inverz módszerek a képszintézisben** | BME, 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

## Tartalom
1. Tenzorok és autograd
2. Lineáris regresszió (kézzel és `nn.Linear`-ral)
3. MNIST osztályozás MLP-vel
4. CIFAR-10 osztályozás konvolúciós hálóval

> **Feladat:** töltsd ki a `# TODO` jelölt részeket!

In [ ]:
# Colab-on szükséges csomagok telepítése (lokálisan általában nem kell)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install torch torchvision matplotlib numpy --quiet


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# Eszköz kiválasztása: GPU ha elérhető, egyébként CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Használt eszköz: {device}')
print(f'PyTorch verzió: {torch.__version__}')


---
## 1. rész – Tenzorok és autograd
A PyTorch alapvető adatszerkezete a **tenzor** (`torch.Tensor`).
NumPy tömbökre hasonlít, de GPU-n is futhat, és automatikus differenciálást (autograd) is támogat.

### 1.1 Tenzor létrehozása, alakja, típusa

In [ ]:
# Különböző módszerek tenzor létrehozására
a = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])
print('Tenzor értéke:\n', a)
print('Alak (shape):  ', a.shape)
print('Típus (dtype): ', a.dtype)
print('Eszköz:        ', a.device)

# Speciális tenzorok
nullak = torch.zeros(3, 4)
egyesek = torch.ones(2, 2)
veletlenszerű = torch.randn(3, 3)
print('\nNullák (3x4):\n', nullak)
print('\nVéletlenszerű (3x3):\n', veletlenszerű)


In [ ]:
# Tenzor átalakítások: reshape, dtype-változtatás, GPU-ra küldés
x = torch.arange(12, dtype=torch.float32)
print('Eredeti:', x.shape)

x_2d = x.reshape(3, 4)
print('Átrendezve (3x4):\n', x_2d)

x_int = x.to(torch.int64)
print('\nEgész típus:', x_int.dtype)

x_gpu = x_2d.to(device)
print(f'\nEszközön: {x_gpu.device}')


### 1.2 Aritmetika és broadcasting

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print('Összeadás:    ', a + b)
print('Szorzás:      ', a * b)
print('Hatványozás:  ', a ** 2)
print('Skaláris szorzat:', torch.dot(a, b))

A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = A @ B
print('\nMátrixszorzás alakja:', C.shape)


In [ ]:
# Broadcasting
m = torch.ones(3, 4)
v = torch.tensor([1., 2., 3., 4.])

eredmeny = m + v
print('Broadcasting eredmény (3x4):\n', eredmeny)

x = torch.randn(100)
print(f'\nÁtlag: {x.mean():.4f}, Szórás: {x.std():.4f}')


### 1.3 Autograd – automatikus differenciálás
A `requires_grad=True` beállítással a PyTorch nyomon követi a műveleteket,
és `.backward()` hívásával kiszámítja a gradienseket.

In [ ]:
# f(x) = x^2 + 3x + 1, f'(x) = 2x + 3
x = torch.tensor(2.0, requires_grad=True)
f = x**2 + 3*x + 1

print(f'f(2) = {f.item():.1f}')  # 11

f.backward()
print(f'df/dx|x=2 = {x.grad.item():.1f}')  # 7


In [ ]:
# Több változós eset
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

x_adat = torch.tensor(3.0)
y_val  = torch.tensor(7.0)

y_hat = w * x_adat + b
loss  = (y_hat - y_val) ** 2

loss.backward()
print(f'loss = {loss.item():.4f}')
print(f'dL/dw = {w.grad.item():.4f}')
print(f'dL/db = {b.grad.item():.4f}')


In [ ]:
# torch.no_grad(): kikapcsoljuk a gradiens-számítást
x = torch.tensor(5.0, requires_grad=True)

with torch.no_grad():
    y = x * 2 + 1
    print('requires_grad kikapcsolva:', y.requires_grad)


---
## 2. rész – Lineáris regresszió
**Feladat:** illesztsük az `y = 2x + 1` egyenest szintetikus, zajos adatokra.

### 2.1 Szintetikus adatok generálása

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

N = 100
x_np = np.linspace(-3, 3, N).astype(np.float32)
y_np = 2.0 * x_np + 1.0 + np.random.randn(N).astype(np.float32) * 0.8

x_tensor = torch.from_numpy(x_np).unsqueeze(1)
y_tensor = torch.from_numpy(y_np).unsqueeze(1)

plt.figure(figsize=(6, 4))
plt.scatter(x_np, y_np, s=15, alpha=0.7, label='Adat')
plt.plot(x_np, 2*x_np + 1, 'r--', label='Igazi egyenes (y=2x+1)')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Szintetikus regressziós adatok')
plt.legend(); plt.tight_layout(); plt.show()


### 2.2 Kézi gradiens-ereszkedés
**TODO:** valósítsd meg a gradiens-ereszkedés lépést és a gradiensek nullázását!

In [ ]:
torch.manual_seed(0)
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.05
n_epochok = 200
veszteségek = []

for epoch in range(n_epochok):
    # Előrevetítés
    y_hat = x_tensor * w + b

    # MSE veszteség
    loss = ((y_hat - y_tensor) ** 2).mean()
    veszteségek.append(loss.item())

    # TODO: töröld a w és b gradienseit (ha már van bennük értéke)
    # Tipp: ellenőrizd, hogy .grad nem None, majd hívj .zero_() metódust
    pass

    # Visszaterjesztés
    loss.backward()

    # TODO: frissítsd w-t és b-t a gradiensek alapján (torch.no_grad() blokkon belül!)
    # Képlet: paraméter = paraméter - lr * paraméter.grad
    pass

print(f'Tanult w = {w.item():.4f}  (igazi: 2.0)')
print(f'Tanult b = {b.item():.4f}  (igazi: 1.0)')

plt.figure(figsize=(6, 3))
plt.plot(veszteségek)
plt.xlabel('Epoch'); plt.ylabel('MSE veszteség')
plt.title('Veszteség alakulása (kézi GD)')
plt.tight_layout(); plt.show()


### 2.3 Lineáris regresszió `nn.Linear`-ral
**TODO:** definiáld a modellt, veszteségfüggvényt és az optimizert, majd írd meg a tanítási hurkot!

In [ ]:
torch.manual_seed(0)

# TODO: hozz létre egy nn.Linear modellt (1 bemenet, 1 kimenet)
model = None  # TODO

# TODO: hozz létre MSELoss veszteségfüggvényt
criterion = None  # TODO

# TODO: hozz létre SGD optimizert (lr=0.05)
optimizer = None  # TODO

veszteségek2 = []
n_epochok = 200

for epoch in range(n_epochok):
    model.train()

    # TODO: 1. Gradiensek nullázása (optimizer segítségével)
    pass

    # TODO: 2. Előrevetítés (model segítségével)
    y_hat = None  # TODO

    # TODO: 3. Veszteség kiszámítása
    loss = None  # TODO
    veszteségek2.append(loss.item())

    # TODO: 4. Visszaterjesztés
    pass

    # TODO: 5. Paraméter frissítés (optimizer segítségével)
    pass

w_tanult = model.weight.item()
b_tanult = model.bias.item()
print(f'Tanult w = {w_tanult:.4f}  (igazi: 2.0)')
print(f'Tanult b = {b_tanult:.4f}  (igazi: 1.0)')


In [ ]:
# Eredmény megjelenítése
model.eval()
with torch.no_grad():
    x_plot = torch.linspace(-3, 3, 100).unsqueeze(1)
    y_pred = model(x_plot).numpy()

plt.figure(figsize=(6, 4))
plt.scatter(x_np, y_np, s=15, alpha=0.6, label='Adatok')
plt.plot(x_np, 2*x_np + 1, 'r--', label='Igazi egyenes')
plt.plot(x_plot.numpy(), y_pred, 'g-', linewidth=2, label=f'Illesztett: {w_tanult:.2f}x + {b_tanult:.2f}')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Lineáris regresszió – illesztett egyenes')
plt.legend(); plt.tight_layout(); plt.show()


---
## 3. rész – MNIST osztályozás MLP-vel
Az MNIST adatbázis 28×28 pixeles, fekete-fehér kézzel írt számjegyeket tartalmaz (0–9).

In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=mnist_transform)
mnist_test  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=mnist_transform)

train_loader_mnist = DataLoader(mnist_train, batch_size=64, shuffle=True,  num_workers=0)
test_loader_mnist  = DataLoader(mnist_test,  batch_size=256, shuffle=False, num_workers=0)

print(f'Tanítóhalmaz mérete: {len(mnist_train)}')
print(f'Teszthalmaz mérete:  {len(mnist_test)}')

kepek, cimkek = next(iter(train_loader_mnist))
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(kepek[i].squeeze(), cmap='gray')
    ax.set_title(str(cimkek[i].item()), fontsize=10)
    ax.axis('off')
plt.suptitle('MNIST minták', fontsize=12)
plt.tight_layout(); plt.show()


### 3.1 MLP modell definíciója
**TODO:** implementáld a `forward()` metódust!

Felépítés: **784 → 256 → 128 → 10**

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        # TODO: hívd meg a self.net hálózatot x-en és add vissza az eredményt
        pass


mlp = MLP().to(device)
print(mlp)
print(f'\nParaméterek száma: {sum(p.numel() for p in mlp.parameters()):,}')


### 3.2 Tanítási és kiértékelési függvények
**TODO:** töltsd ki a tanítási hurok lépéseit és a kiértékelő ciklust!

In [ ]:
def tanit_egy_epoch(model, loader, criterion, optimizer, device):
    """Egy epoch tanítás."""
    model.train()
    ossz_veszteség = 0.0
    helyes = 0
    ossz = 0

    for kepek, cimkek in loader:
        kepek, cimkek = kepek.to(device), cimkek.to(device)

        # TODO: 1. Gradiensek nullázása
        pass

        # TODO: 2. Előrevetítés
        kimenetek = None  # TODO

        # TODO: 3. Veszteség
        loss = None  # TODO

        # TODO: 4. Visszaterjesztés
        pass

        # TODO: 5. Paraméter frissítés
        pass

        ossz_veszteség += loss.item() * kepek.size(0)
        _, josolt = kimenetek.max(1)
        helyes += josolt.eq(cimkek).sum().item()
        ossz   += kepek.size(0)

    return ossz_veszteség / ossz, 100. * helyes / ossz


def kiertekkel(model, loader, criterion, device):
    """Kiértékelés (nincs gradiens-számítás)."""
    model.eval()
    ossz_veszteség = 0.0
    helyes = 0
    ossz = 0

    # TODO: torch.no_grad() blokkon belül menj végig a loader-en,
    # számítsd ki a veszteséget és a pontosságot
    # Útmutató:
    #   - kimenetek = model(kepek)
    #   - loss = criterion(kimenetek, cimkek)
    #   - _, josolt = kimenetek.max(1)
    pass

    return ossz_veszteség / ossz, 100. * helyes / ossz


In [ ]:
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.Adam(mlp.parameters(), lr=1e-3)

N_EPOCH_MNIST = 5
history_mnist = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print('Epoch | Train Loss | Train Acc | Test Loss | Test Acc')
print('-' * 55)
for epoch in range(1, N_EPOCH_MNIST + 1):
    tr_loss, tr_acc = tanit_egy_epoch(mlp, train_loader_mnist, mlp_criterion, mlp_optimizer, device)
    te_loss, te_acc = kiertekkel(mlp, test_loader_mnist, mlp_criterion, device)

    history_mnist['train_loss'].append(tr_loss)
    history_mnist['train_acc'].append(tr_acc)
    history_mnist['test_loss'].append(te_loss)
    history_mnist['test_acc'].append(te_acc)

    print(f'  {epoch:3d}  | {tr_loss:.4f}     | {tr_acc:.2f}%   | {te_loss:.4f}    | {te_acc:.2f}%')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochok = range(1, N_EPOCH_MNIST + 1)

ax1.plot(epochok, history_mnist['train_loss'], label='Tanítás')
ax1.plot(epochok, history_mnist['test_loss'],  label='Teszt')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Veszteség')
ax1.set_title('MNIST – Veszteség'); ax1.legend()

ax2.plot(epochok, history_mnist['train_acc'], label='Tanítás')
ax2.plot(epochok, history_mnist['test_acc'],  label='Teszt')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Pontosság (%)')
ax2.set_title('MNIST – Pontosság'); ax2.legend()

plt.tight_layout(); plt.show()


In [ ]:
mlp.eval()
kepek, cimkek = next(iter(test_loader_mnist))
kepek_gpu = kepek.to(device)

with torch.no_grad():
    kimenetek = mlp(kepek_gpu)
    _, josolt = kimenetek.max(1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(kepek[i].squeeze(), cmap='gray')
    helyes_e = josolt[i].item() == cimkek[i].item()
    szin = 'green' if helyes_e else 'red'
    ax.set_title(f'J:{josolt[i].item()} V:{cimkek[i].item()}', color=szin, fontsize=8)
    ax.axis('off')
plt.suptitle('MLP jóslatok (J=Jósolt, V=Valódi, zöld=helyes, piros=hibás)', fontsize=10)
plt.tight_layout(); plt.show()


---
## 4. rész – CIFAR-10 osztályozás konvolúciós hálóval (CNN)
A CIFAR-10 adatbázis 32×32 pixeles, 10 osztályba sorolt színes képeket tartalmaz.

In [ ]:
CIFAR10_OSZTALYOK = ['repülő', 'autó', 'madár', 'macska', 'szarvas',
                     'kutya', 'béka', 'ló', 'hajó', 'tehergépkocsi']

cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

cifar_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=cifar_train_transform)
cifar_test  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=cifar_test_transform)

train_loader_cifar = DataLoader(cifar_train, batch_size=128, shuffle=True,  num_workers=0, pin_memory=True)
test_loader_cifar  = DataLoader(cifar_test,  batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

print(f'Tanítóhalmaz: {len(cifar_train)} kép')
print(f'Teszthalmaz:  {len(cifar_test)}  kép')


In [ ]:
def denorm(x, mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)):
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    return (x * s + m).clamp(0, 1)

kepek_c, cimkek_c = next(iter(train_loader_cifar))
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    img = denorm(kepek_c[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CIFAR10_OSZTALYOK[cimkek_c[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 minták', fontsize=12)
plt.tight_layout(); plt.show()


### 4.1 Konvolúciós háló definíciója
**TODO:** implementáld a `forward()` metódust!

Felépítés: **2× (Conv-BN-ReLU-MaxPool)** blokk + fully-connected osztályozó fej.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, n_osztaly=10):
        super().__init__()

        # 1. konvolúciós blokk: 3 -> 32 csatorna, 32x32 -> 16x16
        self.blokk1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # 2. konvolúciós blokk: 32 -> 64 csatorna, 16x16 -> 8x8
        self.blokk2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Osztályozó fej: 64*8*8 -> 256 -> n_osztaly
        self.fej = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_osztaly)
        )

    def forward(self, x):
        # TODO: futtasd x-et a blokk1-en, blokk2-n, majd a fejen keresztül
        # és add vissza a végeredményt
        pass


cnn = ConvNet().to(device)
print(cnn)
print(f'\nParaméterek száma: {sum(p.numel() for p in cnn.parameters()):,}')


### 4.2 Tanítási hurok
A `tanit_egy_epoch` és `kiertekkel` függvényeket a 3. részben definiáltuk – itt újra felhasználjuk.

In [ ]:
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
cnn_scheduler = optim.lr_scheduler.StepLR(cnn_optimizer, step_size=10, gamma=0.3)

N_EPOCH_CIFAR = 15
history_cifar = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print('Epoch | Train Loss | Train Acc | Test Loss | Test Acc')
print('-' * 55)
for epoch in range(1, N_EPOCH_CIFAR + 1):
    tr_loss, tr_acc = tanit_egy_epoch(cnn, train_loader_cifar, cnn_criterion, cnn_optimizer, device)
    te_loss, te_acc = kiertekkel(cnn, test_loader_cifar, cnn_criterion, device)
    cnn_scheduler.step()

    history_cifar['train_loss'].append(tr_loss)
    history_cifar['train_acc'].append(tr_acc)
    history_cifar['test_loss'].append(te_loss)
    history_cifar['test_acc'].append(te_acc)

    print(f'  {epoch:3d}  | {tr_loss:.4f}     | {tr_acc:.2f}%   | {te_loss:.4f}    | {te_acc:.2f}%')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochok = range(1, N_EPOCH_CIFAR + 1)

ax1.plot(epochok, history_cifar['train_loss'], label='Tanítás')
ax1.plot(epochok, history_cifar['test_loss'],  label='Teszt')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Veszteség')
ax1.set_title('CIFAR-10 – Veszteség'); ax1.legend()

ax2.plot(epochok, history_cifar['train_acc'], label='Tanítás')
ax2.plot(epochok, history_cifar['test_acc'],  label='Teszt')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Pontosság (%)')
ax2.set_title('CIFAR-10 – Pontosság'); ax2.legend()

plt.tight_layout(); plt.show()


In [ ]:
cnn.eval()
kepek_t, cimkek_t = next(iter(test_loader_cifar))
kepek_t_gpu = kepek_t.to(device)

with torch.no_grad():
    kimenetek_t = cnn(kepek_t_gpu)
    _, josolt_t = kimenetek_t.max(1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = denorm(kepek_t[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    helyes_e = josolt_t[i].item() == cimkek_t[i].item()
    szin = 'green' if helyes_e else 'red'
    ax.set_title(f'{CIFAR10_OSZTALYOK[josolt_t[i].item()]}\n({CIFAR10_OSZTALYOK[cimkek_t[i].item()]})',
                 color=szin, fontsize=7)
    ax.axis('off')
plt.suptitle('CNN jóslatok (felső: jósolt, zárójelben: valódi; zöld=helyes, piros=hibás)', fontsize=10)
plt.tight_layout(); plt.show()
